# W4｜赋值以后，到底改了什么？
在 VS Code 打开本目录并选择 `week-04/.venv` 的 kernel。第一次演示先重启 kernel，随后按实验组运行。
每组独立实验都有自己的初始化；同一组内先修改再观察，不在观察前重新初始化。
本 Notebook 的参考输出用于教师展示与学生复现，不替代 `variant.py` 和自己的测试。
标明 NameError／TypeError 的 cell 是故意的反例，看到预期错误后继续下一组。

## 先看全图：进程内存、名字与对象
Python 程序运行起来是一个进程；解释器在这个进程的内存中保存对象，也维护名字与对象的绑定。Notebook 的 kernel 就是执行代码、维护这些运行状态的进程。
本周把变量理解为用来找到对象的名字。`probe` 是名字，`42` 是整数对象；列表与字典也都是对象，各有身份、类型和值。
```text
当前 Python 进程的内存（关系示意）

名字与绑定                    对象
probe ─────────────────────→ 整数对象：42
```
箭头表示绑定。执行 `名字 = 表达式` 时，先计算右边得到对象，再建立或更新左边名字的绑定；使用名字时，沿当前绑定找到对象。
名字、绑定与对象都由进程维护；图的排布只表示关系，不代表实际内存分区。列表元素、字典条目还能关联其他对象，后面遇到时再沿关系画下去。
整数的值不能原地改写；列表、字典允许修改内容。每行都问：改了哪个名字的绑定，还是哪个对象的内容？还有谁能找到这个对象？

## 从 Bash 到 Notebook：代码在哪里执行？
终端的 Python 进程与 Notebook kernel 是不同的运行进程，各自维护内存中的名字绑定与对象。在新 kernel 查询终端里曾定义的名字，它不会自动出现。下一 cell 故意产生 NameError。

In [1]:
print(probe)

NameError: name 'probe' is not defined

### 初始化 → 第一次观察 → 修改 → 再次观察
分别运行四个 cell。`probe = 42` 建立名字到整数对象的绑定；`probe = 2026` 换了绑定，没有把整数 42 原地改成 2026。最后指着两次观察问：为什么旧输出仍显示 42？

In [2]:
probe = 42

In [3]:
print(probe)

42


In [4]:
probe = 2026

In [5]:
print(probe)

2026


重新运行前面的“第一次观察”，现在应输出 2026；不要先运行初始化。cell 共享同一个 kernel 中的名字绑定与对象；旧输出不会自动刷新。
手动重启 kernel，再只运行任一观察 cell，应为 NameError：原来的运行状态与 probe 绑定已不在。完成后按顺序恢复本组；清空输出只清显示，不会重置运行状态。

## 先认识实验材料：列表、字典与类型
新练习组：列表按位置取值，位置从 0 开始；字典按文字键取值。`type()` 查看对象类型。先按顺序运行，分清整个列表与其中的整数，再沿“名字 → 字典 → 列表 → 整数”读写。
同样写方括号，列表里填位置，字典里填键；`type()` 告诉我们是哪类对象，是不是同一对象要在后面另查。

In [1]:
sizes = [1000, 10000, 100000]
print(sizes[0], sizes[2])
print(type(sizes), type(sizes[2]))

1000 100000
<class 'list'> <class 'int'>


In [2]:
sizes[2] = 500000
print(sizes)

[1000, 10000, 500000]


In [3]:
config = {"seed": 42, "sample_sizes": [1000, 10000, 100000]}
print(type(config))
print(config["seed"])
print(config["sample_sizes"])

<class 'dict'>
42
[1000, 10000, 100000]


In [4]:
print(config["sample_sizes"])
print(config["sample_sizes"][2])
print(type(config["sample_sizes"]), type(config["sample_sizes"][2]))

[1000, 10000, 100000]
100000
<class 'list'> <class 'int'>


In [5]:
config["seed"] = 2026
config["sample_sizes"][2] = 500000
print(config)

{'seed': 2026, 'sample_sizes': [1000, 10000, 500000]}


## 导师要追加实验，旧配置却变了
昨天已用 seed=42 和三组投针数做出论文图表。导师要求保留旧方案，换种子并增加最大投针数。
source 是名字，绑定一个字典对象；字典的 `"seed"`、`"sample_sizes"` 条目分别关联整数与列表。读取 `source["seed"]`，先由名字找到字典，再按键找到整数。
在修改前预测：source 会不会变化？

In [6]:
source = {"seed": 42, "sample_sizes": [1000, 10000, 100000]}

In [7]:
variant = source
print("旧实验：", source)
print("追加实验：", variant)

旧实验： {'seed': 42, 'sample_sizes': [1000, 10000, 100000]}
追加实验： {'seed': 42, 'sample_sizes': [1000, 10000, 100000]}


In [8]:
variant["seed"] = 2026
variant["sample_sizes"][2] = 500000

In [9]:
print("旧实验：", source)
print("追加实验：", variant)

旧实验： {'seed': 2026, 'sample_sizes': [1000, 10000, 500000]}
追加实验： {'seed': 2026, 'sample_sizes': [1000, 10000, 500000]}


## 内容相等，是不是同一对象？
保留上一组的 source、variant，再建立内容相等的第三份配置。`==` 比内容，`is` 比身份。
运行后沿结果画关系：source、variant 指向同一字典 A，same_content 指向字典 B；A、B 是关系图中的对象编号，不是内存地址。`variant = source` 只增加绑定到同一对象的名字，没有复制字典。

In [10]:
same_content = {"seed": 2026, "sample_sizes": [1000, 10000, 500000]}
print(source == variant, source is variant)
print(source == same_content, source is same_content)

True True
True False


## 数字没跟着变：名字重新绑定
下面是新的实验组。先预测：seed 换成 2026，old_seed 会不会一起变？
42 与 2026 都是内存中的整数对象。整数的值不能原地改写；`seed = 2026` 只移动 seed 的箭头，old_seed 的绑定不变。

In [11]:
seed = 42
old_seed = seed
print(seed is old_seed)

True


In [12]:
seed = 2026
print(seed, old_seed, seed is old_seed)

2026 42 False


## 同一列表：重新绑定名字，还是修改内容？
两组从相同的干净初态分别运行，比较改变发生在哪里。
### 第一组：重新绑定 l
先运行初始化，让 l 与 other 关联同一列表；预测后再运行下一 cell，给 l 重新赋值。

In [6]:
l = [1000, 10000, 100000]
other = l

In [7]:
l = [1000, 10000, 500000]
print(l, other, l is other)

[1000, 10000, 500000] [1000, 10000, 100000] False


右边的列表表达式建立新列表 M，l 改为绑定它；other 的绑定没动，仍关联原列表 L。
```text
l     → 新列表 M：[1000, 10000, 500000]
other → 原列表 L：[1000, 10000, 100000]
```
### 第二组：从相同初态重来，修改列表第三项
下一 cell 重新初始化，不承接第一组的新列表；留下 saved_item 对旧整数的引用。预测 other 与 saved_item 的结果；第三项位置是 2。

In [13]:
l = [1000, 10000, 100000]
other = l
saved_item = l[2]
print(l is other)

True


In [14]:
l[2] = 500000
print(l, other, l is other)
print(saved_item, l[2], saved_item is l[2])

[1000, 10000, 500000] [1000, 10000, 500000] True
100000 500000 False


第二组没有移动 l、other 的绑定；`l[2] = 500000` 改的是同一列表的第三项关联，原整数 100000 没有被原地改写。
```text
l ─────┐
       ├→ 同一个列表 L：[1000, 10000, 500000]
other ─┘

saved_item → 整数对象：100000
列表 L 的第三项 → 整数对象：500000
```
对照两组：第一组改 l 的绑定，第二组改列表内容。other 的绑定两次都没有改变；只有第二组通过 other 观察到的内容变了。按这一行的操作目标与共享关系判断结果。

## 复制以后，先分别改两项
新实验组，先只改 seed，再改第三个投针数。不要在两次修改之间重新初始化。

In [15]:
source = {"seed": 42, "sample_sizes": [1000, 10000, 100000]}
variant = source.copy()
print(source is variant)

False


In [16]:
variant["seed"] = 2026
print("旧实验：", source)
print("追加实验：", variant)

旧实验： {'seed': 42, 'sample_sizes': [1000, 10000, 100000]}
追加实验： {'seed': 2026, 'sample_sizes': [1000, 10000, 100000]}


In [17]:
variant["sample_sizes"][2] = 500000
print("旧实验：", source)
print("追加实验：", variant)

旧实验： {'seed': 42, 'sample_sizes': [1000, 10000, 500000]}
追加实验： {'seed': 2026, 'sample_sizes': [1000, 10000, 500000]}


## 沿关系再查一层
接着上一组，检查字典和列表分别是不是同一对象。
字典、列表都是当前 kernel 内存中的对象。字典不同，内部条目仍可能关联同一列表；沿“名字 → 字典 → 列表 → 整数”查共享发生在哪一层。

In [18]:
print(source is variant)
print(source["sample_sizes"] is variant["sample_sizes"])

False
True


## “旧配置没变”的检查也可能被骗
下面三段各自初始化。直接引用、浅复制快照和独立基准分别回答什么？

In [19]:
source = {"seed": 42, "sample_sizes": [1000, 10000, 100000]}
before = source
variant = source
variant["sample_sizes"][2] = 500000
print(source == before)
print(source)

True
{'seed': 42, 'sample_sizes': [1000, 10000, 500000]}


In [20]:
source = {"seed": 42, "sample_sizes": [1000, 10000, 100000]}
before = source.copy()
variant = source
variant["sample_sizes"][2] = 500000
print(source == before, source["sample_sizes"] is before["sample_sizes"])

True True


In [21]:
source = {"seed": 42, "sample_sizes": [1000, 10000, 100000]}
expected_source = {"seed": 42, "sample_sizes": [1000, 10000, 100000]}
variant = source
variant["sample_sizes"][2] = 500000
print(source == expected_source)

False


## 怎样让两版配置真正独立？
新字典配新列表；预期两版内容不同，两层身份也不同。

In [22]:
source = {"seed": 42, "sample_sizes": [1000, 10000, 100000]}
variant = source.copy()
variant["sample_sizes"] = source["sample_sizes"].copy()
variant["seed"] = 2026
variant["sample_sizes"][2] = 500000
print(source)
print(variant)
print(source is variant, source["sample_sizes"] is variant["sample_sizes"])

{'seed': 42, 'sample_sizes': [1000, 10000, 100000]}
{'seed': 2026, 'sample_sizes': [1000, 10000, 500000]}
False False


## 同参数两次调用：内容相等，仍要各自独立
这里使用 examples.py 的已公开参考，不替你实现 variant.py。
预测 first == second、first is second，以及修改 first 后另两份是否保持。

In [23]:
from examples import reference as make_variant
source = {"seed": 42, "sample_sizes": [1000, 10000, 100000]}
first = make_variant(source, 2026, 2, 500000)
second = make_variant(source, 2026, 2, 500000)
print(first == second, first is second,
      first["sample_sizes"] is second["sample_sizes"])

True False False


In [24]:
first["seed"] = 2028
first["sample_sizes"][1] = 250000
print("first:", first)
print("second:", second)
print("source:", source)
assert second == {"seed": 2026, "sample_sizes": [1000, 10000, 500000]}
assert source == {"seed": 42, "sample_sizes": [1000, 10000, 100000]}

first: {'seed': 2028, 'sample_sizes': [1000, 250000, 500000]}
second: {'seed': 2026, 'sample_sizes': [1000, 10000, 500000]}
source: {'seed': 42, 'sample_sizes': [1000, 10000, 100000]}


## 选做：参数没变化，是否仍需要独立？
先执行直接共享故障，观察它能通过内容检查；再修改结果，查看旧配置。

In [25]:
from examples import direct
source = {"seed": 42, "sample_sizes": [1000, 10000, 100000]}
expected_source = {"seed": 42, "sample_sizes": [1000, 10000, 100000]}
result = direct(source, 42, 2, 100000)
print("内容正确:", result == expected_source, source == expected_source)
print("身份共享:", result is source,
      result["sample_sizes"] is source["sample_sizes"])

内容正确: True True
身份共享: True True


In [26]:
result["sample_sizes"][0] = 2000
print(source)
print("旧内容保持:", source == expected_source)

{'seed': 42, 'sample_sizes': [2000, 10000, 100000]}
旧内容保持: False


## 备用：对象地址、堆栈与存储位置
若追问堆与栈：它们涉及具体实现的内存组织与调用管理。CPython 的整数、列表、字典等 Python 对象都由解释器的私有堆管理；名字和绑定的具体存储方式本周不展开。
回到同一列表的两组实验：对象类型相同，操作目标不同，结果就不同。判断依据仍是名字绑定、对象能否修改，以及谁共享这个对象。

## 备用：字符串能原地改字符吗？
下一 cell 故意产生 TypeError，再运行后面的新实验。

In [27]:
s = "cat"
s[1] = "u"

TypeError: 'str' object does not support item assignment

In [28]:
s = "cat"
old = s
s = s.replace("a", "u")
print(s, old, s is old)

cut cat False


## 备用：相同结果，不同的对象关系
两段各自初始化；先预测，再运行。

In [29]:
a = [1, 2]
b = a
a += [3]
print(a, b, a is b)

[1, 2, 3] [1, 2, 3] True


In [30]:
a = [1, 2]
b = a
a = a + [3]
print(a, b, a is b)

[1, 2, 3] [1, 2] False


In [31]:
a = [1, 2]
b = a
a[0] = 9
a = a + [3]
print(a, b, a is b)

[9, 2, 3] [9, 2] False


## 回到本周作业
独立编写 tests/test_student.py，先让同一份测试接受参考、拒绝 direct/shallow，再实现 variant.py。
说明一条断言的依据及实际共享关系。Notebook 展示不等于你自己的函数和测试已完成。